In [265]:
import sqlite3
import pandas as pd

con = sqlite3.connect(r'C:\Users\jazee\OneDrive\Desktop\airtel-telecom-analytics\data\raw\airtel.db')

def q(sql):
    return pd.read_sql(sql, con)


In [266]:
q("""
SELECT COUNT(*)
FROM customer_features;
""")

,COUNT(*)
0,22000


In [267]:
q("""
SELECT ROUND(AVG(actual_arpu_6m), 2)
FROM customer_features;
""")

,"ROUND(AVG(actual_arpu_6m), 2)"
0,344.46


In [268]:
q("""
SELECT SUM(complaint_count = 0)
FROM customer_features;
""")

,SUM(complaint_count = 0)
0,6631


In [269]:
q("""
SELECT ab_group, COUNT(*)
FROM customer_features
GROUP BY ab_group;
""")

,ab_group,COUNT(*)
0,A,10996
1,B,11004


In [270]:
# A1 - Count subscribers by connection type

q("""
SELECT
    connection_type,
    COUNT(*) AS subs,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM customer_features),
        2
    ) AS pct
FROM customer_features
GROUP BY connection_type
ORDER BY subs DESC;
""")

,connection_type,subs,pct
0,Prepaid,17149,77.95
1,Postpaid,4851,22.05


In [271]:
# A2 - Popular Plans

q("""
SELECT
    plan_name,
    COUNT(*) AS subscribers
FROM customer_features
GROUP BY plan_name
ORDER BY subscribers DESC
LIMIT 10;
""")

,plan_name,subscribers
0,Prepaid Annual 1799,2901
1,Prepaid Quarterly 719,2561
2,Prepaid Max 479,2392
3,Prepaid Entertainment 399,2142
4,Prepaid Data 349,1884
5,Prepaid Popular 299,1699
6,Prepaid Value 209,1437
7,Postpaid Infinity 1499,1431
8,Postpaid Family 999,1239
9,Prepaid Smart 155,1137


In [272]:
q("""
SELECT
    circle_name,
    COUNT(*) AS subscribers
FROM customer_features
GROUP BY circle_name
ORDER BY subscribers DESC
LIMIT 10;
""")

,circle_name,subscribers
0,Maharashtra,1428
1,Andhra Pradesh,1412
2,Mumbai,1395
3,Karnataka,1361
4,North East,1318
5,Assam,1301
6,Himachal Pradesh,1247
7,Gujarat,1210
8,Kerala,1141
9,Rajasthan,1082


In [273]:
# A4 - Subscribers by Age Group

q("""
SELECT
    age_band,
    COUNT(*) AS subscribers,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM customer_features),
        2
    ) AS percentage
FROM customer_features
GROUP BY age_band
ORDER BY subscribers DESC;
""")

,age_band,subscribers,percentage
0,25-34,7467,33.94
1,18-24,5268,23.95
2,35-44,4824,21.93
3,45-54,2895,13.16
4,55+,1546,7.03


In [274]:
# A5 - Subscribers by Acquisition Channel

q("""
SELECT
    acquisition_channel,
    COUNT(*) AS subscribers,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM customer_features),
        2
    ) AS percentage
FROM customer_features
GROUP BY acquisition_channel
ORDER BY subscribers DESC;
""")

,acquisition_channel,subscribers,percentage
0,Retailer,9287,42.21
1,Airtel App,6513,29.60
2,Website,3081,14.00
3,Call Center,1770,8.05
4,Doorstep,1349,6.13


In [275]:
# A6 - Device and Operating System Analysis

q("""
SELECT
    device_type,
    os,
    COUNT(*) AS subscribers,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM customer_features),
        2
    ) AS percentage
FROM customer_features
GROUP BY device_type, os
ORDER BY subscribers DESC;
""")

,device_type,os,subscribers,percentage
0,Smartphone,Android,15613,70.97
1,Smartphone,iOS,3358,15.26
2,Feature Phone,Other,3029,13.77


In [276]:
# A7 - Subscriber Tenure
q("""
SELECT
    CASE
        WHEN tenure_days < 180 THEN 'Below 6 Months'
        WHEN tenure_days < 365 THEN '6-12 Months'
        WHEN tenure_days < 730 THEN '1-2 Years'
        WHEN tenure_days < 1095 THEN '2-3 Years'
        ELSE '3+ Years'
    END AS tenure_group,
    COUNT(*) AS subscribers
FROM customer_features
GROUP BY tenure_group
ORDER BY subscribers DESC;
""")

,tenure_group,subscribers
0,2-3 Years,7055
1,1-2 Years,6961
2,6-12 Months,3526
3,3+ Years,3374
4,Below 6 Months,1084


In [277]:
# B1 - Average stated ARPU

q("""
SELECT
    ROUND(AVG(stated_arpu), 2) AS avg_arpu,
    COUNT(*) AS subs
FROM customer_features;
""")

,avg_arpu,subs
0,421.65,22000


In [278]:
# B2 - Average actual ARPU and monthly total

q("""
SELECT
    ROUND(AVG(actual_arpu_6m), 2) AS avg_actual,
    ROUND(SUM(actual_arpu_6m), 0) AS monthly_total
FROM customer_features;
""")

,avg_actual,monthly_total
0,344.46,7578210.0


In [279]:
# B3 - Difference between stated and actual ARPU

q("""
SELECT
    ROUND(AVG(stated_arpu), 2) AS stated,
    ROUND(AVG(actual_arpu_6m), 2) AS actual,
    ROUND(AVG(arpu_gap), 2) AS gap,
    ROUND(
        100.0 * (SUM(stated_arpu) - SUM(actual_arpu_6m))
        / NULLIF(SUM(actual_arpu_6m), 0), 2
    ) AS over_pct
FROM customer_features;
""")

,stated,actual,gap,over_pct
0,421.65,344.46,77.18,22.41


In [280]:
# B4 - ARPU by connection type

q("""
SELECT
    connection_type,
    COUNT(*) AS subs,
    ROUND(AVG(stated_arpu), 2) AS stated,
    ROUND(AVG(actual_arpu_6m), 2) AS actual
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,subs,stated,actual
0,Postpaid,4851,959.65,970.42
1,Prepaid,17149,269.46,167.40


In [281]:
# B5 - Top 5 plans by actual monthly revenue

q("""
SELECT
    plan_name,
    COUNT(*) AS subs,
    ROUND(SUM(actual_arpu_6m), 0) AS rev,
    ROUND(AVG(actual_arpu_6m), 2) AS per_sub
FROM customer_features
GROUP BY plan_name
ORDER BY rev DESC
LIMIT 5;
""")

,plan_name,subs,rev,per_sub
0,Postpaid Infinity 1499,1431,2171756.0,1517.65
1,Postpaid Family 999,1239,1249246.0,1008.27
2,Postpaid 749,946,718959.0,760.00
3,Prepaid Entertainment 399,2142,671850.0,313.66
4,Prepaid Data 349,1884,520066.0,276.04


In [282]:
# B6 - Top 5 circles by actual monthly revenue

q("""
SELECT
    circle_name,
    COUNT(*) AS subs,
    ROUND(SUM(actual_arpu_6m), 0) AS rev
FROM customer_features
GROUP BY circle_name
ORDER BY rev DESC
LIMIT 5;
""")

,circle_name,subs,rev
0,Mumbai,1395,473288.0
1,Karnataka,1361,468515.0
2,Andhra Pradesh,1412,463370.0
3,North East,1318,460998.0
4,Maharashtra,1428,450032.0


In [283]:
# B7 - Payments and amount received Jan-Jun 2025

q("""
SELECT
    COUNT(*) AS payments,
    ROUND(SUM(amount_inr) / 10000000.0, 2) AS crore_inr
FROM recharges
WHERE recharge_date BETWEEN '2025-01-01'
                        AND '2025-06-30';
""")

,payments,crore_inr
0,83693,4.55


In [284]:
# B8 - Actual ARPU by tenure

q("""
SELECT
    tenure_years,
    COUNT(*) AS subs,
    ROUND(AVG(actual_arpu_6m), 2) AS arpu
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
""")

,tenure_years,subs,arpu
0,0,4610,329.10
1,1,6961,342.72
2,2,7055,350.13
3,3,3374,357.22


In [285]:
# B9 - Subscribers with no recharge and churn

q("""
SELECT
    SUM(CASE WHEN recharge_count = 0 THEN 1 ELSE 0 END)
        AS paid_nothing,
    ROUND(
        100.0 * AVG(
            CASE
                WHEN recharge_count = 0 THEN is_churned
            END
        ), 2
    ) AS their_churn,
    ROUND(AVG(stated_arpu), 2) AS base_stated_arpu
FROM customer_features;
""")

,paid_nothing,their_churn,base_stated_arpu
0,3711,44.3,421.65


In [286]:
# ===== C1 — Payments and distinct subscribers by txn_type =====
q("""
SELECT txn_type,
       COUNT(*) AS payments,
       COUNT(DISTINCT customer_id) AS subs
FROM recharges
GROUP BY txn_type;
""")

,txn_type,payments,subs
0,Bill Payment,108419,4851
1,Recharge,299670,17149


In [287]:
# ===== C2 — Overall average payment (all rows) =====
q("""
SELECT ROUND(AVG(amount_inr), 2) AS avg_payment
FROM recharges;
""")

,avg_payment
0,532.59


In [288]:
# ===== C3 — Average, min, max payment by txn_type =====
q("""
SELECT txn_type,
       ROUND(AVG(amount_inr), 2) AS avg_amt,
       MIN(amount_inr) AS mn,
       MAX(amount_inr) AS mx
FROM recharges
GROUP BY txn_type;
""")

,txn_type,avg_amt,mn,mx
0,Bill Payment,1082.42,399,1873
1,Recharge,333.66,99,1799


In [289]:
# ===== C4 — Average recharge count and total by connection type =====
q("""
SELECT connection_type,
       ROUND(AVG(recharge_count), 2) AS payments,
       ROUND(AVG(recharge_total_6m), 0) AS total_6m
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,payments,total_6m
0,Postpaid,5.39,5823.0
1,Prepaid,3.36,1004.0


In [290]:
# ===== C5 — Payment channels by transaction count =====
q("""
SELECT channel,
       COUNT(*) AS payments,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM recharges), 1) AS pct
FROM recharges
GROUP BY channel
ORDER BY payments DESC;
""")

,channel,payments,pct
0,Website,111163,27.2
1,Airtel App,111083,27.2
2,USSD,75257,18.4
3,Retailer,74406,18.2
4,AutoPay,36180,8.9


In [291]:
# ===== C6 — Channel share within each txn_type =====
q("""
SELECT txn_type,
       channel,
       COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY txn_type), 1) AS pct
FROM recharges
GROUP BY txn_type, channel
ORDER BY txn_type, n DESC;
""")

,txn_type,channel,n,pct
0,Bill Payment,Airtel App,36186,33.4
1,Bill Payment,AutoPay,36180,33.4
2,Bill Payment,Website,36053,33.3
3,Recharge,USSD,75257,25.1
4,Recharge,Website,75110,25.1
5,Recharge,Airtel App,74897,25.0
6,Recharge,Retailer,74406,24.8


In [292]:
# ===== C7 — Churn by recharge_count (8 or fewer) =====
q("""
SELECT recharge_count,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn_pct
FROM customer_features
WHERE recharge_count <= 8
GROUP BY recharge_count
ORDER BY recharge_count;
""")

,recharge_count,subs,churn_pct
0,0,3711,44.30
1,1,2762,40.04
2,2,2339,28.35
3,3,1026,54.09
4,4,714,76.05
5,5,3407,13.77
6,6,6732,1.52
7,7,507,5.52
8,8,32,90.63


In [293]:
# ===== D1 — Monthly data traffic (TB) and active subscribers =====
q("""
SELECT month,
       COUNT(DISTINCT customer_id) AS active_subs,
       ROUND(SUM(total_mb) / 1024.0 / 1024.0, 1) AS total_tb
FROM v_usage_month
GROUP BY month
ORDER BY month;
""")

,month,active_subs,total_tb
0,2025-01,21302,338.0
1,2025-02,21162,307.3
2,2025-03,20366,332.5
3,2025-04,19320,310.3
4,2025-05,18356,363.1
5,2025-06,17326,338.9


In [294]:
# ===== D2 — Average data_mb across usage_daily rows =====
q("""
SELECT ROUND(AVG(data_mb), 2) AS avg_mb,
       COUNT(*) AS rows_used
FROM usage_daily;
""")

,avg_mb,rows_used
0,1572.87,1326739


In [295]:
# ===== D3 — Average MB per active day vs per observed day =====
q("""
SELECT ROUND(AVG(mb_per_active_day), 2) AS per_active_day,
       ROUND(AVG(mb_per_observed_day), 2) AS per_observed_day
FROM customer_features;
""")

,per_active_day,per_observed_day
0,1404.44,524.05


In [296]:
# ===== D4 — Average daily data by weekday =====
q("""
SELECT strftime('%w', usage_date) AS dow,
       COUNT(*) AS rows_used,
       ROUND(AVG(data_mb), 1) AS avg_mb
FROM usage_daily
GROUP BY dow
ORDER BY dow;
""")

,dow,rows_used,avg_mb
0,0,190368,1578.1
1,1,189914,1572.9
2,2,183823,1569.5
3,3,191088,1564.5
4,4,190764,1580.0
5,5,190542,1569.5
6,6,190240,1575.5


In [297]:
# ===== D5 — Distinct active subscribers by month =====
q("""
SELECT month,
       COUNT(DISTINCT customer_id) AS active_subs
FROM v_usage_month
GROUP BY month
ORDER BY month;
""")

,month,active_subs
0,2025-01,21302
1,2025-02,21162
2,2025-03,20366
3,2025-04,19320
4,2025-05,18356
5,2025-06,17326


In [298]:
# ===== D6 — Average voice minutes and SMS by month (Jan–Jun 2025) =====
q("""
SELECT strftime('%Y-%m', usage_date) AS month,
       ROUND(AVG(voice_minutes), 1) AS avg_voice,
       ROUND(AVG(sms_count), 3) AS avg_sms
FROM usage_daily
WHERE usage_date BETWEEN '2025-01-01' AND '2025-06-30'
GROUP BY month
ORDER BY month;
""")

,month,avg_voice,avg_sms
0,2025-01,55.6,0.745
1,2025-02,55.8,0.746
2,2025-03,56.1,0.750
3,2025-04,56.5,0.760
4,2025-05,56.7,0.764
5,2025-06,57.1,0.772


In [299]:
# ===== D7 — Monthly data per subscriber (GB) and average usage days =====
q("""
SELECT month,
       ROUND(AVG(total_mb) / 1024.0, 2) AS avg_gb_per_sub,
       ROUND(AVG(usage_days), 1) AS avg_days
FROM v_usage_month
GROUP BY month
ORDER BY month;
""")

,month,avg_gb_per_sub,avg_days
0,2025-01,16.25,11.4
1,2025-02,14.87,10.3
2,2025-03,16.72,11.5
3,2025-04,16.44,11.2
4,2025-05,20.26,11.7
5,2025-06,20.03,11.5


In [300]:
# ===== D8 — Monthly MB per active day for never-churned subscribers =====
q("""
SELECT m.month,
       ROUND(AVG(m.total_mb * 1.0 / NULLIF(m.usage_days, 0)), 1) AS mb_per_day
FROM v_usage_month AS m
JOIN customer_features AS c
  ON c.customer_id = m.customer_id
WHERE c.is_churned = 0
GROUP BY m.month
ORDER BY m.month;
""")

,month,mb_per_day
0,2025-01,1410.6
1,2025-02,1422.1
2,2025-03,1418.1
3,2025-04,1411.8
4,2025-05,1659.5
5,2025-06,1662.2


In [301]:
# ===== E1 — Four quartiles of daily data use (mb_per_active_day) =====
q("""
WITH n AS (
    SELECT mb_per_active_day AS m,
           NTILE(4) OVER (ORDER BY mb_per_active_day) AS quartile
    FROM customer_features
    WHERE usage_days > 0
)
SELECT quartile,
       COUNT(*) AS subs,
       ROUND(MIN(m), 0) AS min_mb,
       ROUND(MAX(m), 0) AS max_mb
FROM n
GROUP BY quartile
ORDER BY quartile;
""")

,quartile,subs,min_mb,max_mb
0,1,5499,25.0,657.0
1,2,5499,657.0,1093.0
2,3,5499,1093.0,1572.0
3,4,5499,1572.0,9720.0


In [302]:
# ===== E2 — Churn by daily data-use quartile =====
q("""
WITH n AS (
    SELECT is_churned,
           NTILE(4) OVER (ORDER BY mb_per_active_day) AS quartile
    FROM customer_features
    WHERE usage_days > 0
)
SELECT quartile,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn_pct
FROM n
GROUP BY quartile
ORDER BY quartile;
""")

,quartile,subs,churn_pct
0,1,5499,30.28
1,2,5499,28.11
2,3,5499,22.51
3,4,5499,14.09


In [303]:
# ===== E3 — Voice and data per active day by connection_type and device_type =====
q("""
SELECT connection_type,
       device_type,
       COUNT(*) AS subs,
       ROUND(AVG(total_voice_min * 1.0 / NULLIF(usage_days, 0)), 1) AS voice_per_day,
       ROUND(AVG(mb_per_active_day), 0) AS mb_per_day
FROM customer_features
WHERE usage_days > 0
GROUP BY connection_type, device_type
ORDER BY voice_per_day DESC;
""")

,connection_type,device_type,subs,voice_per_day,mb_per_day
0,Postpaid,Feature Phone,690,48.6,531.0
1,Postpaid,Smartphone,4161,48.0,3353.0
2,Prepaid,Feature Phone,2338,47.9,166.0
3,Prepaid,Smartphone,14807,47.9,1093.0


In [304]:
# ===== E4 — Monthly allowance vs actual monthly use by plan =====
q("""
SELECT plan_name,
       monthly_gb,
       ROUND(AVG(total_data_mb) / 1024.0 / 6, 2) AS actual_gb_per_month
FROM customer_features
WHERE usage_days > 0
GROUP BY plan_name, monthly_gb
ORDER BY monthly_gb;
""")

,plan_name,monthly_gb,actual_gb_per_month
0,Prepaid Lite 99,15.0,2.88
1,Prepaid Smart 155,30.0,5.72
2,Prepaid Value 209,30.0,5.89
3,Postpaid 399,40.0,7.86
4,Prepaid Max 479,45.0,8.84
5,Prepaid Popular 299,45.0,8.93
6,Prepaid Annual 1799,60.0,11.87
7,Prepaid Quarterly 719,60.0,11.78
8,Postpaid 499,75.0,15.47
9,Prepaid Data 349,75.0,15.13


In [305]:
# ===== E5 — Allowance usage and subscribers below half allowance =====
q("""
SELECT ROUND(AVG(monthly_gb), 1) AS allowance_gb,
       ROUND(AVG(total_data_mb / 1024.0 / 6), 2) AS used_gb,
       ROUND(100.0 * AVG((total_data_mb / 1024.0 / 6) / NULLIF(monthly_gb, 0)), 1) AS pct_of_allowance,
       SUM((total_data_mb / 1024.0 / 6) < monthly_gb * 0.5) AS under_half
FROM customer_features
WHERE usage_days > 0;
""")

,allowance_gb,used_gb,pct_of_allowance,under_half
0,76.8,15.44,19.8,20560


In [306]:
# ===== E6 — Spend, data use and churn by monetary score (built with NTILE) =====
q("""
WITH scored AS (
    SELECT recharge_total_6m,
           mb_per_active_day,
           is_churned,
           NTILE(4) OVER (ORDER BY recharge_total_6m) AS m_score
    FROM customer_features
)
SELECT m_score,
       COUNT(*) AS subs,
       ROUND(AVG(recharge_total_6m), 0) AS spend_6m,
       ROUND(AVG(mb_per_active_day), 0) AS mb_per_day,
       ROUND(100.0 * AVG(is_churned), 2) AS churn_pct
FROM scored
GROUP BY m_score
ORDER BY m_score;
""")

,m_score,subs,spend_6m,mb_per_day,churn_pct
0,1,5500,167.0,973.0,54.82
1,2,5500,945.0,778.0,19.96
2,3,5500,1678.0,1117.0,11.20
3,4,5500,5477.0,2750.0,9.04


In [307]:
# ===== E7 — Churn by usage_days quartile =====
q("""
WITH n AS (
    SELECT usage_days,
           is_churned,
           NTILE(4) OVER (ORDER BY usage_days) AS quartile
    FROM customer_features
)
SELECT quartile,
       COUNT(*) AS subs,
       MIN(usage_days) AS min_days,
       MAX(usage_days) AS max_days,
       ROUND(100.0 * AVG(is_churned), 2) AS churn_pct
FROM n
GROUP BY quartile
ORDER BY quartile;
""")

,quartile,subs,min_days,max_days,churn_pct
0,1,5500,0,31,58.56
1,2,5500,31,57,24.45
2,3,5500,57,86,9.49
3,4,5500,86,180,2.51


In [308]:
# ===== E8 — SMS per active day and subscribers who never sent SMS =====
q("""
SELECT ROUND(AVG(total_sms * 1.0 / NULLIF(usage_days, 0)), 3) AS sms_per_active_day,
       SUM(total_sms = 0) AS never_sent_one,
       COUNT(*) AS subs
FROM customer_features
WHERE usage_days > 0;
""")

,sms_per_active_day,never_sent_one,subs
0,0.598,654,21996


In [309]:
# ===== F1 — Roaming vs non-roaming days and distinct subscribers =====
q("""
SELECT is_roaming,
       COUNT(*) AS days,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM usage_daily), 2) AS pct,
       COUNT(DISTINCT customer_id) AS subs
FROM usage_daily
GROUP BY is_roaming;
""")

,is_roaming,days,pct,subs
0,0,1287143,97.02,21994
1,1,39596,2.98,16233


In [310]:
# ===== F2 — Average data and voice on roaming vs non-roaming days =====
q("""
SELECT is_roaming,
       ROUND(AVG(data_mb), 2) AS avg_mb,
       ROUND(AVG(voice_minutes), 2) AS avg_voice
FROM usage_daily
GROUP BY is_roaming;
""")

,is_roaming,avg_mb,avg_voice
0,0,1572.95,56.27
1,1,1570.05,56.09


In [311]:
# ===== F3 — Roaming by prepaid and postpaid subscribers =====
q("""
SELECT connection_type,
       COUNT(*) AS subs,
       ROUND(AVG(roaming_days), 2) AS avg_roam_days,
       ROUND(100.0 * AVG(roaming_days > 0), 1) AS pct_ever_roamed
FROM customer_features
WHERE usage_days > 0
GROUP BY connection_type;
""")

,connection_type,subs,avg_roam_days,pct_ever_roamed
0,Postpaid,4851,1.84,74.8
1,Prepaid,17145,1.79,73.5


In [312]:
# ===== F4 — Churn by roaming-day bands =====
q("""
SELECT CASE
           WHEN roaming_days = 0 THEN '0'
           WHEN roaming_days <= 2 THEN '1-2'
           WHEN roaming_days <= 5 THEN '3-5'
           ELSE '6+'
       END AS band,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn_pct
FROM customer_features
WHERE usage_days > 0
GROUP BY band
ORDER BY band;
""")

,band,subs,churn_pct
0,0,5763,44.61
1,1-2,9922,21.58
2,3-5,5546,8.69
3,6+,765,3.92


In [313]:
# ===== F5 — Top 5 circles by roaming share of usage days =====
q("""
SELECT circle_name,
       ROUND(100.0 * SUM(roaming_days) / NULLIF(SUM(usage_days), 0), 2) AS pct_roaming
FROM customer_features
WHERE usage_days > 0
GROUP BY circle_name
ORDER BY pct_roaming DESC
LIMIT 5;
""")

,circle_name,pct_roaming
0,Odisha,3.07
1,Haryana,3.07
2,Delhi,3.07
3,Karnataka,3.05
4,Assam,3.05
